# Methods — an equation-by-equation audit

*Companion to Part II — Metabolic Scaling Theory and Biological Fractals.* CC-BY-4.0. This is a **reviewer's** notebook: it exists to test the paper's claims, not just to illustrate them.

Every check below uses **exact rational arithmetic** (`fractions.Fraction`) — no floating point, no extra dependencies — so the algebra is verifiable. Addresses MAJOR issues #1 and #2.

In [1]:
import sys, os
HERE = os.getcwd()
if HERE not in sys.path:
    sys.path.insert(0, HERE)
# fractal_review_utils.py lives next to these notebooks. Launch Jupyter from
# docs/notebooks/metabolic-scaling/review/ (or add that folder to sys.path).
import numpy as np
import matplotlib.pyplot as plt
import fractal_review_utils as u
print('utils loaded from', u.__file__)
from fractions import Fraction as F
import math

utils loaded from /Users/tswetnam/github/fractal-notebooks/docs/notebooks/metabolic-scaling/review/fractal_review_utils.py


## 1. The WBE branching ratios (Eq 10) — these are correct
`ξ = r_{k+1}/r_k = n^{-1/2}` (area-preserving) and `γ = l_{k+1}/l_k = n^{-1/3}` (space-filling). For binary branching `n = 2`:

In [2]:
n = 2
xi_v = n ** (-1/2); gamma_v = n ** (-1/3)
print(f'xi  = n^(-1/2) = {xi_v:.4f}  (area-preserving, large vessels)')
print(f'gamma = n^(-1/3) = {gamma_v:.4f}  (space-filling)')
# Space-filling Hausdorff dimension of the *network*:
D_network = math.log(n) / math.log(1/gamma_v)
print(f'D_network = ln n / ln(1/gamma) = {D_network:.4f}   <-- the genuine WBE value is 3')

xi  = n^(-1/2) = 0.7071  (area-preserving, large vessels)
gamma = n^(-1/3) = 0.7937  (space-filling)
D_network = ln n / ln(1/gamma) = 3.0000   <-- the genuine WBE value is 3


## 2. The metabolic exponent (from the *correct* chain) — 3/4, exactly
With radius exponent `a = 1/2` and length exponent `b = 1/3`, `θ = 1/(2a + b)`:

In [3]:
a, b = F(1,2), F(1,3)
theta = 1 / (2*a + b)
print('theta = 1/(2a+b) =', theta, '=', float(theta))
print('equivalently D/(D+1) with D=3 :', F(3,1)/(F(3,1)+1), '=', float(F(3,4)))
assert theta == F(3,4)
print('OK: the 3/4 law is internally consistent -- when derived from D=3.')

theta = 1/(2a+b) = 3/4 = 0.75
equivalently D/(D+1) with D=3 : 3/4 = 0.75
OK: the 3/4 law is internally consistent -- when derived from D=3.


## 3. Where the printed Methods derivation breaks
Now the projection argument the paper uses to claim `d = 3/2`.

In [4]:
issues = []

# Eq 12: sphere volume printed as (4/3) pi l^(2/3); a sphere is (4/3) pi r^3.
printed_exp = F(2,3); correct_exp = F(3,1)
issues.append(('Eq 12 sphere volume exponent',
               f'printed l^{printed_exp}, geometry requires l^{correct_exp}; '
               f'also contradicts the line above it (v_n proportional to l^3)'))

# Eq 16: A ~ V_B^{1/2} l^{3/2} r, with V_B ~ L^3 => exponent sum:
exp_sum = F(1,2)*3 + F(3,2) + F(1,1)   # V_B^{1/2}=L^{3/2}, l^{3/2}, r^1
issues.append(('Eq 16 dimensional check',
               f'RHS scales as L^{exp_sum}; an AREA must scale as L^2. '
               f'Off by L^{exp_sum - 2}.'))

# Eq 17: box count must go as N(eps) ~ eps^{-D} (negative power).
issues.append(('Eq 17 sign of the exponent',
               'printed N(eps) ~ eps^{+3/2}; a box count REQUIRES eps^{-D}. '
               'A positive power means fewer boxes as boxes shrink (impossible).'))
issues.append(('Eq 17 stated limit vs result',
               '"dim -> 2" and "d = 3/2" are asserted together; only one can hold.'))

for name, msg in issues:
    print(f'- {name}:\n    {msg}\n')

- Eq 12 sphere volume exponent:
    printed l^2/3, geometry requires l^3; also contradicts the line above it (v_n proportional to l^3)

- Eq 16 dimensional check:
    RHS scales as L^4; an AREA must scale as L^2. Off by L^2.

- Eq 17 sign of the exponent:
    printed N(eps) ~ eps^{+3/2}; a box count REQUIRES eps^{-D}. A positive power means fewer boxes as boxes shrink (impossible).

- Eq 17 stated limit vs result:
    "dim -> 2" and "d = 3/2" are asserted together; only one can hold.



## 4. Reconciling 3 vs 3/2 vs 4/3
These are dimensions of **different objects**. The paper must pick the object it is measuring and state the matching prediction:

In [5]:
targets = {
  'full 3-D network (Hausdorff)': F(3,1),
  '2-D projection, self-affine mass dim (paper: 3/2)': F(3,2),
  '2-D projection, alt derivation (paper Results: 4/3)': F(4,3),
}
for k, v in targets.items():
    print(f'{v!s:>4} = {float(v):.4f}   {k}')
print()
print('The Methods claim d=3/2 is cited to "West et al., unpublished" and the')
print('4/3 in Results cites an "Equation 18" that does not appear in the text.')
print('A referee cannot verify either until the derivation is shown in full.')

   3 = 3.0000   full 3-D network (Hausdorff)
 3/2 = 1.5000   2-D projection, self-affine mass dim (paper: 3/2)
 4/3 = 1.3333   2-D projection, alt derivation (paper Results: 4/3)

The Methods claim d=3/2 is cited to "West et al., unpublished" and the
4/3 in Results cites an "Equation 18" that does not appear in the text.
A referee cannot verify either until the derivation is shown in full.


### Reviewer note
The *area-preserving + space-filling* core (Eq 10 → `D = 3`, `θ = 3/4`) checks out exactly. The **projection argument** that yields the paper's headline `3/2` does not, as printed. Fix the projection derivation (or replace it with the correct closed form) and choose **one** target before comparing to data.